In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv; load_dotenv()
import os

/home/pawan/anaconda3/envs/langchain/lib/python3.13/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
/home/pawan/anaconda3/envs/langchain/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1",
    task="text-generation",
    huggingfacehub_api_token=os.getenv('HF_TOKEN')
)
model = ChatHuggingFace(llm=llm)

In [3]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [4]:
graph = StateGraph(BlogState)

In [5]:
def create_outline(state: BlogState) -> BlogState:

    title = state['title']
    prompt = f"Generate an outline from the provided title: {title}"
    state['outline'] = model.invoke(prompt).content

    return state

def generate_content(state: BlogState) -> BlogState:

    outline = state['outline']
    prompt = f"Generate a detailed blog based on the provided outline: {outline}"
    state['content'] = model.invoke(prompt).content
    
    return state

graph.add_node('create_outline', create_outline)
graph.add_node('generate_content', generate_content)

In [6]:
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'generate_content')
graph.add_edge('generate_content', END)

workflow = graph.compile()

In [7]:
initial_state = {'title': 'DOOM Game'}
workflow.invoke(initial_state)

{'title': 'DOOM Game',
 'outline': '\n### Comprehensive Outline: **DOOM Game**  \n\n#### **I. Introduction**  \n- **A. Overview of DOOM**:  \n  - Revolutionary first-person shooter (FPS) released by id Software in 1993.  \n  - Pioneered 3D graphics, networked multiplayer, and modding culture.  \n- **B. Historical Context**:  \n  - Emergence during the PC gaming boom; influenced by *Wolfenstein 3D*.  \n  - Defined the "fast-paced shooter" genre.  \n- **C. Thesis Statement**:  \n  - *DOOM* redefined video games through technological innovation, cultural impact, and enduring legacy.  \n\n#### **II. Development and Release**  \n- **A. id Software and Key Creators**:  \n  - John Carmack (engine programming), John Romero (design), Adrian Carmack (art).  \n- **B. Technological Breakthroughs**:  \n  - **id Tech 1 Engine**: Raycasting, non-orthogonal levels, dynamic lighting.  \n  - Optimization for low-spec hardware.  \n- **C. Release Strategy**:  \n  - Shareware model (Episode 1 free); full g